## Choresky-Algorithmus 
Ziel ist es die Matrix $A$ in zwei Dreiecksmatrizen zu zerlegen, um das LGS $Ax=b$ zu lösen. Diese können dann nacheinander gelöst werden, was die Effizienz der Rechnung verbessert. Die Matrix $A$ muss allerdings symmetrisch und positiv definit sein. Symmetrisch bedeutet, dass $A^T = A$. Posotif definit bedeutet, dass alle Eigenwerte strikt größer null sind.

In [59]:
import numpy as np 

# Beispiel-Matrix 
A = np.array([[4, 2, 4, 0], 
			  [2, 2, 3, 2], 
			  [4, 3, 9, 4],
			  [0, 2, 4, 14]], dtype=float)

# Auf Symmetrie und positive Definitheit prüfen
if not np.allclose(A, A.T):
    print('Nicht symmetrisch')
else:
    eigenwerte = np.linalg.eigvals(A)
    if eigenwerte.all() > 0:
        print('Matrix symmetrisch und positiv definit')
    else:
        print('Matrix symmetrisch, aber nicht positiv definit')

Matrix symmetrisch und positiv definit


Wenn die Probe positiv ist kann mithilfe des Cholesky Algorithmus die untere Dreiecksmatrix aufgestellt werden

In [ ]:
n = A.shape[0] 
L = np.zeros_like(A) # Wir erstellen eine leere Matrix für das Ergebnis

for k in range(n): 
	for i in range(k, n): 
		if i == k: # Diagonalelemente 
		# Hier nutzt du dein Slicing: Summe der Quadrate der bereits berechneten L-Werte 
			summe = np.sum(L[k, :k]**2) 
			L[k, k] = np.sqrt(A[k, k] - summe) 
		else: # Elemente unterhalb der Diagonale # Hier nutzt du das Skalarprodukt der bisherigen Zeilenanteile 
			summe = np.sum(L[i, :k] * L[k, :k]) 
			L[i, k] = (A[i, k] - summe) / L[k, k] 

print(f'Matrix L\n{L}')
print(L.T)

Matrix L
[[2. 0. 0. 0.]
 [1. 1. 0. 0.]
 [2. 1. 2. 0.]
 [0. 2. 1. 3.]]
[[2. 1. 2. 0.]
 [0. 1. 1. 2.]
 [0. 0. 2. 1.]
 [0. 0. 0. 3.]]


Anschließend kann eine Probe durch geführt werden.  
Da $A=LL^T$ entspricht, wird die transponierte Matrix von $L$ gebildet und A berchnet

In [46]:
# Probe 
if np.allclose(A, L @ L.T):
    print('Es hat funktioniert!')
else:
    print('Es gibt einen Fehler!')

Es hat funktioniert!


Das LGS kann dann durch Vorwärts- und Rückwärtseinsetzen gelöst werden. Zuerst wird ein Hilfsvektor $y= L^T\cdot x$ definiert mit dem die erste Matrix gelöst werden kann. Da $L$ eine untere Dreiecksmatrix ist kann man $y_1$ direkt ablesen und die weiteren Werte nach unten durch einsetzen bestimmen. In Python kann eine solche Matrix mit ```scipy.linalg.solve_triangular``` berechnet werden. Diese Funktion ist für Dreiecksmatrizen optimiert.

In [47]:
from scipy.linalg import solve_triangular

# vorgebener Vektor b
b = np.array([4, 9, 13, 42])

# 1. Ergebnisvektor Ly = b berechnen
y = solve_triangular(L, b, lower=True)

# 2. Ergebnisvektor L.T x = y berechnen
x = solve_triangular(L.T, y, lower=False)

# Ergebnisvektor x
print(f'Ergebnis: {x}')

Ergebnis: [ 1.  2. -1.  3.]


Zusätzlich kann noch eine Probe durchgeführt werden ob $A \cdot x = b$ wahr ist

In [48]:
# Ergebnisprobe, A * x muss gleich b sein
if np.allclose(A @ x, b): 
    print('Ergebnis ist True')
else:
    print('Ergebnis ist False')

Ergebnis ist True


In Python kann die Cholesky-Zerlegung auch als Numpy Funktion ausgeführt werden. Dazu wird die Funktion ```np.linalg.cholesky``` verwendet. 

In [60]:
from numpy.linalg import cholesky

L_easy = cholesky(A)

print(np.allclose(L_easy, L))

True
